# **Odkrywanie Wiedzy i Systemy Rekomendacyjne** - *Laboratorium 4*
## Predykcja oceny z wykorzystaniem pakietu ***surprise***

In [ ]:
# Instalacja pakietu surprise
!pip install --upgrade pandas==2.1.4
!pip install --upgrade numpy==1.26.4
!pip install scikit-surprise

#Wykonaj polecenie **Środowisko wykonawcze -> Uruchom ponownie sesję**.

---



In [ ]:
# Wczytanie bibliotek
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from surprise import Dataset, AlgoBase, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split, GridSearchCV, cross_validate

W ramach pakietu **surprise** możemy w prosty sposób operować na zbiorze **MovieLens**.

In [ ]:
data = Dataset.load_builtin("ml-100k")

Poniższy przykład pokazuje najprostszy sposób oceny błędu dla algorytmu **SVD** z wykorzystaniem walidacji skrośnej.

In [ ]:
algo = SVD()

cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=10, verbose=True)

## Zadanie 1. (1 pkt)
Wykonaj walidację skrośną dla algorytmu UserKNN z kosinusową miarą podobieństwa.

In [ ]:
#Miejsce na rozwiązanie




---

Oprócz walidacji skrośnej, również z pakietem **surprise** możemy dzielić dane na podzbiory trenigowy i testowy.

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2)

Wybrany algorytm należy wytrenować na zbiorze treningowym.

In [ ]:
algo = SVD()
algo.fit(trainset)

Po czym możemy wygenerować predykcje i ocenić je zgodnie z wybraną miarą.

In [ ]:
predictions = algo.test(testset)

rmse = accuracy.rmse(predictions)

## Zadanie 2. (0,5 pkt)
Wyznacz wartość średniego błędu bezwzględnego dla algorytmu SVD.

In [ ]:
# Miejsce na rozwiązanie


## Zadanie 3. (2 pkt)
1.   Wyznacz wartości błędów dla algorytmów ItemKNN i UserKNN z podobieństwem kosinusowym na tych samych danych, co dla algorytmu SVD.
2.   Co możesz powiedzieć o działaniu algorytmów? Czy któryś z nich jest lepszy? Dlaczego?





In [ ]:
# Miejsce na odpowiedź 1.


*Miejsce na odpowiedź 2.*



> Tutaj



---



Pakiet **surprise** pozwala również na automatyzację strojenia parametrów algorytmu. Do tego celu służy klasa *GridSearchCV*. Przykład automatycznego doboru parametrów dla algorytmu **UserKNN** został przedstawiony poniżej.

**Uwaga!** Kod wykonuje się ok. 4 min.


In [ ]:
param_grid = {
  'k': [10, 20, 30, 40, 50], # liczba najbliższych sąsiadów
  'sim_options': {
      'name': ['cosine', 'msd', 'pearson'], # miary podobieństwa, które będą brane pod uwagę
      'user_based': [True] # czy podobieństwo użytkowników, czy itemów - w praktyce oznacza wersję UserKNN lub ItemKNN
  }
}
gs = GridSearchCV(KNNBasic, param_grid, measures=['rmse', 'mae'], cv=5) # ustawienie miar błędów oraz ilość foldów dla walidacji skrośnej
gs.fit(data)

Obiekty klasy *GridSearchCV* zawierają słownik *cv_result*, który zawiera wartości miar błędów dla wszystkich podziałów walidacji skrośnej dla wszystkich kombinacji wartości parametrów.

In [ ]:
results_df = pd.DataFrame(gs.cv_results)

# spłaszczenie słownika sim_options dla łatwiejszego tworzenia wykresów
results_df['sim_metric'] = results_df['param_sim_options'].apply(lambda x: x.get('name', None) if isinstance(x, dict) else None)
results_df['user_based'] = results_df['param_sim_options'].apply(lambda x: x.get('user_based', None) if isinstance(x, dict) else None)

print(results_df.head())

In [ ]:
# Pobieranie unikatowych wartości parametrów
unique_metrics = results_df['sim_metric'].unique()
unique_user_based = results_df['user_based'].unique()

# Tworzenie wykresu dla każdej z miar podobieństwa
num_metrics = len(unique_metrics)
fig, axes = plt.subplots(num_metrics, 1, figsize=(10, 5 * num_metrics), constrained_layout=True)

if num_metrics == 1:
    axes = [axes]

for ax, metric in zip(axes, unique_metrics):
    for user_based in unique_user_based:
        filtered = results_df[
            (results_df['sim_metric'] == metric) &
            (results_df['user_based'] == user_based)
        ]

        if filtered.empty:
            print(f"No data for Metric={metric}, User-Based={user_based}")
            continue

        ax.plot(
            filtered['param_k'],
            filtered['mean_test_rmse'],
            label=f"UB: {user_based}"
        )

    ax.set_title(f'Hyperparameter Tuning for {metric} Similarity')
    ax.set_xlabel('Number of Neighbors (k)')
    ax.set_ylabel('RMSE')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.suptitle('Hyperparameter Tuning Results')
plt.savefig("params_tuning.jpg")
plt.show()

In [ ]:
print(f"Best params: {gs.best_params['rmse']}")
print(f"Best RMSE: {gs.best_score['rmse']}")

## Zadanie 4. (2,5 pkt)
1. Narysuj analogiczne, co powyżej wykresy dla
*   MAE,
*   czasu.
2. Co możesz powiedzieć o otrzymanych wynikach? Czy któraś konfiguracja jest zawsze optymalna? Którą miarą najlepiej się kierować?


In [ ]:
# Miejsce na odpowiedź.

*Miejsce na odpowiedź 2.*



> Tutaj



---



## Zadanie 5 (1,5 pkt)
Znajdź optymalne hiperparametry dla algorytmu ItemKNN względem miar błędu i czasu. Zwizualizuj wyniki analizy na wykresie.

In [ ]:
# Miejsce na rozwiązanie.


## Zadanie 6. (2,5 pkt)
1.   Napisz własną implementację naiwnego algorytmu najwyżej oceniane.
2.   Porównaj wyniki z wyznaczoną wcześniej najlepszą wersją algorytmu kNN. Co możesz powiedzieć o tych wynikach?



In [ ]:
# Miejsce na odpowiedź 1.


*Miejsce na odpowiedź 2.*



> Tutaj



---

